In [ ]:
import os

from dotenv import load_dotenv
from mc_postgres_db.models import Asset, Provider, ProviderAssetMarket
from mc_postgres_db.operations import set_data
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)

with Session(engine) as session:
    stmt = select(Provider).where(Provider.name == "Kraken")
    kraken = session.execute(stmt).scalar_one()
    display(kraken)

In [ ]:
import pandas as pd
from sqlalchemy import select

to_asset_name = "FET"
from_asset_name = "USD"

with Session(engine) as session:
    stmt = select(Provider).where(Provider.name == "Kraken")
    provider = session.execute(stmt).scalar_one()

    print(provider)

    stmt = select(Asset).where(Asset.name == to_asset_name)
    to_asset = session.execute(stmt).scalar_one()

    print(to_asset)

    stmt = select(Asset).where(Asset.name == from_asset_name)
    from_asset = session.execute(stmt).scalar_one()

    print(from_asset)

    path = f"/Users/glynfinck/Downloads/Kraken_OHLCVT/{to_asset.name}{from_asset.name}_1.csv"

    # Read CSV without headers
    df = pd.read_csv(path, header=None)

    # Set meaningful column names for OHLCV data
    df.columns = ["timestamp", "open", "high", "low", "close", "volume", "trade_count"]

    # Format the timestamp to be a datetime object
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
    df["provider_id"] = provider.id
    df["from_asset_id"] = from_asset.id
    df["to_asset_id"] = to_asset.id
    df["open"] = df["open"].astype(float)
    df["high"] = df["high"].astype(float)
    df["low"] = df["low"].astype(float)
    df["close"] = df["close"].astype(float)
    df["volume"] = df["volume"].astype(float)
    df["trade_count"] = df["trade_count"].astype(int)

    df.drop(columns=["trade_count"], inplace=True)

# Display the first few rows of the dataframe
df

In [ ]:
with Session(engine) as session:
    stmt = (
        select(ProviderAssetMarket)
        .where(
            ProviderAssetMarket.provider_id == kraken.id,
            ProviderAssetMarket.to_asset_id == to_asset.id,
            ProviderAssetMarket.from_asset_id == from_asset.id,
        )
        .limit(100)
    )
    sample_df = pd.read_sql(stmt, engine)
    display(sample_df)

In [ ]:
from tqdm.notebook import tqdm

# Split into batches and upsert into the database.
chunk_size = 100000
total_rows = len(df)

print(f"Processing {total_rows} rows in batches of {chunk_size}")

for i in tqdm(range(0, total_rows, chunk_size)):
    chunk = df.iloc[i : i + chunk_size]
    batch_num = (i // chunk_size) + 1
    total_batches = (total_rows + chunk_size - 1) // chunk_size
    try:
        set_data(
            engine, ProviderAssetMarket.__tablename__, chunk, operation_type="upsert"
        )
    except Exception as e:
        print(f"Error processing batch {batch_num}: {e}")
        # You might want to break here or continue depending on your needs
        break

print("Batch processing complete!")